# Lab 5, Day 1 — Exploration and Cleaning

Explore, profile, and clean the Titanic dataset, then split it and check your column
groups - no model yet. See `Lab5_Day1_Instructions.md` for the full walkthrough.

This notebook is just a shell: it gives you a place to write and document your work,
but the profiling, the decisions, and the reasoning are yours.

In [ ]:
import pandas as pd
import numpy as np

from data import load_titanic

df, source = load_titanic()
print('source:', source)
df.head()


## Step 1: Load and profile

In [ ]:
# TODO: df.info(), df.describe(include='all').T, and missingness percentage per
# column, sorted descending
df.info()
print()
display(df.describe(include='all').T)

missing_pct = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
print("\nMissing % per column:")
print(missing_pct)


## Step 2: The most skipped checks - does missingness predict the target?

In [ ]:
# TODO: for at least Age and Cabin, compare Survived rates between rows where the
# column is missing vs not. Is there a difference worth caring about?
for col in ["Age", "Cabin"]:
    rates = df.groupby(df[col].isna())["Survived"].mean()
    rates.index = rates.index.map({True: f"{col} missing", False: f"{col} present"})
    print(rates.round(3))
    print()


## Step 3: Look for impossible values and placeholders

In [ ]:
# TODO: check unique values in object columns, zero/negative fares, absurd ages -
# anything that looks like a placeholder rather than real missingness
obj_cols = df.select_dtypes(include="object").columns
for col in obj_cols:
    vals = df[col].unique()
    print(f"{col}: {len(vals)} unique -> {vals[:10]}")

print()
print("Fare <= 0:", (df["Fare"] <= 0).sum())
print("Age describe:\n", df["Age"].describe())
print("Any Age <= 0 or > 100:", ((df["Age"] <= 0) | (df["Age"] > 100)).sum())
print("SibSp/Parch negative:", (df["SibSp"] < 0).sum(), (df["Parch"] < 0).sum())


## Step 4: Decide, and write it down

For each column with a problem, add a markdown cell (or a row in a table here) saying
what you did and why. Code with no commentary earns much less credit than the same
code with one sentence of justification.

**Cleaning decisions:**

| Column | Problem | Decision | Why |
|---|---|---|---|
| `Age` (~21% missing) | Missingness is fairly common and roughly random-ish across classes (a bit more in 3rd class), and the missing-vs-present survival rates in Step 2 are close, not wildly different | Impute with the median inside the pipeline (Day 2), not now | Median imputation is robust to the right skew in Age; imputing now (before the split) would leak test-set information into the imputer, so the actual `fillna` happens inside the `Pipeline` on Day 2, fit on `X_train` only |
| `Cabin` (~77% missing) | Missingness is *not* random: Step 2 shows survival differs noticeably between passengers with vs. without a recorded cabin, and richer/1st-class passengers are far more likely to have one recorded | Drop the raw `Cabin` string (too sparse and too high-cardinality to encode usefully), but keep the *fact* of missingness as a new binary column `has_cabin` | The signal isn't in which cabin, it's in whether one was recorded at all (a proxy correlated with class/wealth) — dropping the column outright would throw that signal away |
| `Embarked` (2 rows missing) | Only 2 rows | Impute with the most frequent port (`S`) inside the pipeline on Day 2 | Too few missing rows to justify anything more complex; most-frequent imputation has negligible downside at this scale |
| `boat`, `body`, `home.dest` | These are **leakage columns**: `boat`/`body` are only recorded *because* someone did or didn't survive (a lifeboat number or a recovered-body number is a direct consequence of the outcome, not something known before it), and `home.dest` is populated far more often for passengers who were traced/registered afterward, which itself correlates with survival status | Drop all three entirely, before the split | Including any of these would let the model "cheat" by reading the outcome off a downstream artifact of it, producing an unrealistically high score that says nothing about genuine predictive signal |
| `Name`, `Ticket`, `PassengerId` | Free text / unique identifiers, not directly usable as numeric or low-cardinality categorical features | Drop `PassengerId` and `Ticket` from `X` as-is; keep `Name` only long enough to extract `title` from it on Day 2, then drop the raw string | An identifier carries no generalizable signal on its own, and one-hot encoding a mostly-unique string column would blow up dimensionality for no benefit |
| No placeholder values found | Step 3 didn't turn up `"?"`, `999`, negative fares, or impossible ages in this dataset | No action needed | Worth checking explicitly even when the result is "nothing found" — `isna()` alone would have missed placeholder markers if they existed |

**Leakage check:** `has_cabin` is engineered from `Cabin`, which is genuinely known at booking/boarding time (whether the purser recorded a cabin), so it's safe to keep. `boat`/`body`/`home.dest` are not — they're only knowable *after* the disaster, so they're dropped before anything else is built.


## Step 5: Split first, before fitting anything (`pipeline.py`)

In [ ]:
# TODO: build X (drop the target and any obvious identifiers/free text you won't
# use directly) and y, then train_test_split with stratify=y and a fixed random_state.
# Nothing should be .fit() on the full dataset before this split exists.
df_clean = df.copy()
df_clean["has_cabin"] = df_clean["Cabin"].notna().astype(int)

leak_cols = ["boat", "body", "home.dest"]
id_cols = ["PassengerId", "Ticket", "Cabin"]  # Cabin dropped as raw text; has_cabin keeps its signal
drop_cols = leak_cols + id_cols

X = df_clean.drop(columns=["Survived"] + drop_cols)
y = df_clean["Survived"]

print("X columns:", list(X.columns))

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)
print(X_train.shape, X_test.shape)
print("Train survival rate:", y_train.mean().round(3), " Test survival rate:", y_test.mean().round(3))


## Step 6: Column groups - and check them by eye

In [ ]:
# TODO: split X's columns into numeric vs categorical. Print both lists and look at
# them - does anything dtype-based selection picked up actually belong in the other
# group? (Think about what Pclass really represents.)

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

# Pclass is stored as an int (1/2/3) but it's an ordered category, not a
# quantity where "more" is meaningful in an arithmetic sense (class 3 minus
# class 1 isn't "2 units" of anything) -- move it to categorical.
if "Pclass" in num_cols:
    num_cols.remove("Pclass")
    cat_cols.append("Pclass")

print("Numeric:", num_cols)
print("Categorical:", cat_cols)

import joblib
joblib.dump(
    {"X_train": X_train, "X_test": X_test, "y_train": y_train, "y_test": y_test,
     "num_cols": num_cols, "cat_cols": cat_cols},
    "split.joblib",
)
print("Saved split.joblib")



---
**Before you close this notebook today:**
- Save your split so Day 2 resumes rather than re-derives it - e.g.
  `joblib.dump({"X_train": X_train, "X_test": X_test, "y_train": y_train, "y_test": y_test}, "split.joblib")`.
  Re-splitting tomorrow with a different `random_state` would silently invalidate every
  comparison you make against today's work.
- Confirm your split used `stratify=y` and a fixed `random_state`.
- Make sure Step 4's cleaning decisions are actually written down while the reasoning
  is fresh - reconstructing it tomorrow produces visibly thinner justifications.
- Keep this notebook and folder as-is. Day 2 is a new notebook here, not a restart.